In [ ]:
# ============================================================
# TIODF KB Probe Runner (Robustness Check)
# ============================================================
# Uses existing KB probe questions from primary analysis.
# No probe generation step needed.
#
# Cell 2  Imports and client setup
# Cell 3  Upload files
# Cell 4  Collect probe responses from robustness subjects
# Cell 5  KB-gap analysis
# Cell 6  Save and download
# ============================================================
!pip install openai pandas -q

In [ ]:
# ============================================================
# Cell 2 - Imports and client setup
# ============================================================
from openai import OpenAI
import pandas as pd
import json, re, time, io
from datetime import datetime
from google.colab import files, userdata

OPENR  = userdata.get('OPENR')
client = OpenAI(api_key=OPENR, base_url="https://openrouter.ai/api/v1")

MODELS = {
    "Gemini-3.1-Pro" : "google/gemini-3.1-pro-preview",
    "Qwen3.6-Max"    : "qwen/qwen3.6-max-preview",
    "Claude-Sonnet"  : "anthropic/claude-sonnet-4-6"
}

MODEL_ORIGIN = {
    "Gemini-3.1-Pro" : "US",
    "Qwen3.6-Max"    : "China",
    "Claude-Sonnet"  : "US"
}

PROBE_MAX_TOKENS = {
    "Gemini-3.1-Pro" : {"Chinese": 300, "English": 300},
    "Qwen3.6-Max"    : {"Chinese": 300, "English": 300},
    "Claude-Sonnet"  : {"Chinese": 300, "English": 300}
}

# Condition order for analysis tables
CONDITION_ORDER = [
    ("Gemini-3.1-Pro", "US",    "Gemini-ZH", "Chinese"),
    ("Gemini-3.1-Pro", "US",    "Gemini-EN", "English"),
    ("Qwen3.6-Max",    "China", "Qwen-ZH",   "Chinese"),
    ("Qwen3.6-Max",    "China", "Qwen-EN",   "English"),
    ("Claude-Sonnet",  "US",    "Claude-ZH", "Chinese"),
    ("Claude-Sonnet",  "US",    "Claude-EN", "English")
]

DIM_MAP = {
    'trans_border'       : 'TB',
    'identity'           : 'ID',
    'cultural_continuity': 'CC',
    'narrative'          : 'NR'
}

print(f"Client initialized")
print(f"Subjects : {list(MODELS.keys())} (OpenRouter)")

In [ ]:
# ============================================================
# Cell 3 - Upload files
#
# Upload THREE files:
#   (1) existing KB probes JSON  (from primary analysis)
#   (2) robustness scored CSV    (for KB-gap merge)
#   (3) robustness raw CSV       (for community name)
#
# KB probes JSON schema (must have these keys per probe):
#   prompt_id, probe_zh, probe_en, correct_answer, knowledge_basis
# ============================================================

print("Upload THREE files:")
print("  (1) existing KB probes JSON (e.g. DaiThai_kb_probes_draft.json)")
print("  (2) robustness scored CSV   (e.g. DaiThai_robustness_scored.csv)")
print("  (3) robustness raw CSV      (e.g. DaiThai_robustness_raw.csv)")

uploaded = files.upload()

probes         = None
scored_df      = None
community_name = "unknown"

for fname, content in uploaded.items():
    if fname.endswith('.json'):
        raw_text  = content.decode('utf-8')
        raw_clean = re.sub(r'^```(?:json)?\s*|\s*```$', '',
                           raw_text, flags=re.DOTALL).strip()
        probes = json.loads(raw_clean)
        for p in probes:
            p['correct_answer'] = 'Yes'  # enforce
        community_name = re.sub(r'[_-]kb.*$', '', fname.replace('.json', ''))
        print(f"\nProbes JSON loaded : {fname}")
        print(f"  {len(probes)} probes")
        for p in sorted(probes, key=lambda x: x['prompt_id']):
            print(f"  {p['prompt_id']}  EN: {p['probe_en'][:65]}")

    elif fname.endswith('.csv') and 'scored' in fname.lower():
        scored_df = pd.read_csv(io.BytesIO(content), on_bad_lines='skip')
        if community_name == "unknown":
            community_name = re.sub(r'[_-]?(robustness[_-])?scored.*$', '',
                                    fname.replace('.csv', ''))
        print(f"\nScored CSV loaded  : {fname}  ({len(scored_df)} rows)")

    elif fname.endswith('.csv') and ('raw' in fname.lower() or 'response' in fname.lower()):
        # raw CSV used only for community name fallback
        if community_name == "unknown":
            community_name = fname.split('_')[0]
        print(f"\nRaw CSV loaded     : {fname}")

assert probes    is not None, "ERROR: KB probes JSON not found"
assert scored_df is not None, "ERROR: Robustness scored CSV not found"

# Detect score columns
SCORE_COLS = {}
for dim, short in DIM_MAP.items():
    for candidate in [dim, f"{dim}_score"]:
        if candidate in scored_df.columns:
            SCORE_COLS[short] = candidate
            break
if 'total_score' in scored_df.columns:
    SCORE_COLS['total'] = 'total_score'

print(f"\nCommunity    : {community_name}")
print(f"Score cols   : {list(SCORE_COLS.values())}")
print(f"Models in scored CSV : {scored_df['model'].unique().tolist()}")

In [ ]:
# ============================================================
# Cell 4 - Collect probe responses from robustness subjects
# 11 probes x 3 models x 2 languages = 66 API calls
# temperature=0  |  subjects: Gemini, Qwen, Claude
# ============================================================

def normalize_answer(text: str) -> str:
    if not text or not isinstance(text, str):
        return "Unknown"
    t = text.strip()
    if re.match(r'^(yes\b|是)', t[:8], re.IGNORECASE):
        return "Yes"
    if re.match(r'^(no\b|否|不是|不对)', t[:8], re.IGNORECASE):
        return "No"
    if re.search(r'\b(yes|是)\b', t[:20], re.IGNORECASE):
        return "Yes"
    if re.search(r'\b(no|否|不是)\b', t[:20], re.IGNORECASE):
        return "No"
    return "Unknown"


def run_probe(question: str, model_id: str, model_name: str,
              language: str, max_retries: int = 3) -> str:
    max_tok = PROBE_MAX_TOKENS[model_name][language]
    for attempt in range(max_retries):
        try:
            resp = client.chat.completions.create(
                model=model_id,
                messages=[{"role": "user", "content": question}],
                temperature=0,
                max_tokens=max_tok,
                extra_headers={
                    "HTTP-Referer": "https://github.com/ooodddee/Trans-border-Representation-Probe",
                    "X-Title": "TIODF KB Probe - Robustness"
                }
            )
            return resp.choices[0].message.content.strip()
        except Exception as e:
            if attempt < max_retries - 1:
                print(f"  Retry {attempt+1}/{max_retries}: {e}")
                time.sleep(5)
            else:
                return f"ERROR: {e}"


probe_results = []
total_calls   = len(probes) * len(MODELS) * 2
current       = 0

print("=" * 65)
print(f"TIODF KB Probe Collection - {community_name} (Robustness)")
print(f"{len(probes)} probes x {len(MODELS)} models x 2 languages = {total_calls} calls")
print(f"temperature=0  |  correct_answer=Yes (by design)")
print("=" * 65)

for probe in sorted(probes, key=lambda x: x['prompt_id']):
    pid = probe['prompt_id']
    for model_name, model_id in MODELS.items():
        for lang, question in [("Chinese", probe['probe_zh']),
                               ("English", probe['probe_en'])]:
            current += 1
            print(f"[{current:02d}/{total_calls}] {pid} | {model_name:<20} | {lang}")

            raw_answer     = run_probe(question, model_id, model_name, lang)
            result         = normalize_answer(raw_answer)
            probe_accepted = (result == probe['correct_answer'])

            probe_results.append({
                "community"      : community_name,
                "prompt_id"      : pid,
                "category"       : pid[0],
                "model"          : model_name,
                "model_origin"   : MODEL_ORIGIN[model_name],
                "language"       : lang,
                "probe_question" : question,
                "raw_answer"     : raw_answer,
                "probe_result"   : result,
                "correct_answer" : probe['correct_answer'],
                "probe_accepted" : probe_accepted,
                "knowledge_basis": probe.get('knowledge_basis', ''),
                "timestamp"      : datetime.now().isoformat()
            })

            tag = "OK" if result == "Yes" else ("KL-DISTORTION" if result == "No" else "Unknown")
            print(f"  -> [{result}] {tag:<24} raw: {raw_answer[:40]}")
            time.sleep(0.5)

probes_df = pd.DataFrame(probe_results)
yes_rate  = (probes_df['probe_result'] == 'Yes').mean()
kl_rate   = (probes_df['probe_result'] == 'No').mean()
print(f"\nCollection complete: {len(probes_df)} responses")
print(f"  Yes (knowledge present) : {yes_rate:.0%}")
print(f"  No  (KL-distortion)     : {kl_rate:.0%}")

In [ ]:
# ============================================================
# Cell 5 - KB-gap analysis
# ============================================================

LOW_SCORE_THRESHOLD = 7

print("=" * 65)
print(f"KB Probe - Results Overview  |  {community_name} (Robustness)")
print("=" * 65)

# Merge narrative scores from scored_df
score_rename     = {v: k for k, v in SCORE_COLS.items()}
score_cols_in_df = [c for c in SCORE_COLS.values() if c in scored_df.columns]
merge_keys       = ['prompt_id', 'model', 'language']
scored_sub       = scored_df[merge_keys + score_cols_in_df].copy().rename(columns=score_rename)
merged           = probes_df.merge(scored_sub, on=merge_keys, how='left')

has_scores = 'total' in merged.columns
print(f"\nMerged rows    : {len(merged)}")
print(f"With scores    : {merged['total'].notna().sum() if has_scores else 'N/A'}")

# Probe pass rate
print("\nProbe pass rate per condition:")
pass_rate = {}
for model_name, origin, label, lang in CONDITION_ORDER:
    sub = merged[(merged['model'] == model_name) & (merged['language'] == lang)]
    if len(sub) == 0:
        continue
    rate = (sub['probe_result'] == 'Yes').mean()
    pass_rate[label] = rate
    bar = '█' * int(rate * 20)
    print(f"  {label:<12}  {rate:.1%}  {bar}")

# KL-distortion
kl_cases = merged[merged['probe_result'] == 'No'][['prompt_id', 'model', 'language']]
print(f"\nKL-distortion ({len(kl_cases)} cases):")
for _, r in kl_cases.iterrows():
    print(f"  {r['prompt_id']:<4} | {r['model']:<22} | {r['language']}")
if len(kl_cases) == 0:
    print("  none")

# Asymmetric probes
pivot_probe = merged.pivot_table(
    index='prompt_id', columns=['model', 'language'],
    values='probe_result', aggfunc='first'
)
asymmetric = []
for pid in pivot_probe.index:
    vals = {}
    for model_name, origin, label, lang in CONDITION_ORDER:
        try:
            vals[label] = pivot_probe.loc[pid, (model_name, lang)]
        except Exception:
            vals[label] = None
    if len(set(v for v in vals.values() if v)) > 1:
        asymmetric.append({'prompt_id': pid, **vals})

print(f"\nAsymmetric probes ({len(asymmetric)} prompts):")
for a in asymmetric:
    vals_str = ' | '.join(f"{l}={a.get(l,'—')}" for _, _, l, _ in CONDITION_ORDER)
    print(f"  {a['prompt_id']:<4}  {vals_str}")
if not asymmetric:
    print("  none")

# Narrative scores (probe=Yes only)
print(f"\nNarrative scores  probe=Yes only  (max=12):")
print(f"  {'cond':<12} {'n':>3}  TB    ID    CC    NR   total")
score_by_condition = {}
for model_name, origin, label, lang in CONDITION_ORDER:
    sub = merged[
        (merged['model'] == model_name) &
        (merged['language'] == lang) &
        (merged['probe_result'] == 'Yes')
    ]
    if len(sub) == 0:
        print(f"  {label:<12}   0  -")
        continue
    row = {'n': len(sub)}
    for dim in ['TB', 'ID', 'CC', 'NR', 'total']:
        row[dim] = round(sub[dim].mean(), 2) if dim in sub.columns else '-'
    score_by_condition[label] = row
    print(f"  {label:<12} {row['n']:>3}  "
          f"{row['TB']:<5} {row['ID']:<5} {row['CC']:<5} {row['NR']:<5} {row['total']}")

# KB-gap cases
kb_gap_cases = []
if has_scores:
    gap_rows = merged[
        (merged['probe_result'] == 'Yes') &
        (merged['total'].notna()) &
        (merged['total'] <= LOW_SCORE_THRESHOLD)
    ]
    for _, r in gap_rows.iterrows():
        kb_gap_cases.append({
            'prompt_id': r['prompt_id'], 'model': r['model'], 'language': r['language'],
            **{d: r.get(d, '-') for d in ['TB', 'ID', 'CC', 'NR', 'total']}
        })

print(f"\nKB-gap cases  (probe=Yes, total<={LOW_SCORE_THRESHOLD})  -  {len(kb_gap_cases)}:")
for c in kb_gap_cases:
    dims = f"TB={c['TB']} ID={c['ID']} CC={c['CC']} NR={c['NR']}"
    print(f"  {c['prompt_id']:<4} | {c['model']:<22} | {c['language']:<8} | {dims} | total={c['total']}")
if not kb_gap_cases:
    print("  none")

In [ ]:
# ============================================================
# Cell 6 - Save and download
# ============================================================

probe_fname = f"{community_name}_robustness_kb_probes_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
probes_df.to_csv(probe_fname, index=False, encoding='utf-8-sig')
print(f"Saved: {probe_fname}  ({len(probes_df)} rows)")
files.download(probe_fname)

summary = {
    "community"           : community_name,
    "timestamp"           : datetime.now().isoformat(),
    "n_total"             : int(len(probes_df)),
    "low_score_threshold" : LOW_SCORE_THRESHOLD,
    "probe_pass_rate"     : pass_rate,
    "kb_gap_cases"        : kb_gap_cases,
    "score_by_condition"  : score_by_condition
}
summary_fname = f"{community_name}_robustness_summary.json"
with open(summary_fname, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
print(f"Saved: {summary_fname}")
files.download(summary_fname)